In [1]:
CSV_PATH = r"C:\Users\nuzha\IDD_project\IDD_DS\NF-ToN-IoT.csv"
OUT_DIR = "fbhesn_extra_results"
SEEDS = [42, 43, 44, 45, 46]
OMEGA_MIN, OMEGA_MAX = 0.3, 1.0        
MAX_EPOCHS = 40
RUN_CONTROLS = True                    # Ctrl-A and Ctrl-B
RUN_BASELINES = True                   # GRU-64, DNN, FB-HESN-128 (slowest part; set False to skip)
STRICT_EQUIVALENCE = True              # stop if the generic layer does not reproduce your layers
QUICK_TEST = False                     # True = tiny smoke test (2 seeds, 3 epochs)

N_PAIRS, TIME_REPS, LAT_CALLS = 20, 5, 200
if QUICK_TEST:
    SEEDS, MAX_EPOCHS, N_PAIRS, TIME_REPS, LAT_CALLS = SEEDS[:2], 3, 4, 2, 20
    OUT_DIR = OUT_DIR + "_quick"          # never mixed with the real run

In [2]:
import os, json, time, random, warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Dropout, Input, Flatten, GRU
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix
from imblearn.over_sampling import RandomOverSampler
warnings.filterwarnings('ignore')
os.makedirs(OUT_DIR, exist_ok=True)
RESUME_KEY = dict(omega=[OMEGA_MIN, OMEGA_MAX], max_epochs=MAX_EPOCHS, csv=CSV_PATH, quick=QUICK_TEST)
_key_path = os.path.join(OUT_DIR, 'resume_key.json')
if os.path.exists(_key_path):
    with open(_key_path) as fh:
        if json.load(fh) != RESUME_KEY:
            raise RuntimeError(f"{OUT_DIR} was produced with different settings (omega interval, epochs, CSV or quick "
                               f"mode). Delete that folder or change OUT_DIR so old results are not mixed in.")
else:
    with open(_key_path, 'w') as fh:
        json.dump(RESUME_KEY, fh)


def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)


set_seed(42)
print("TensorFlow:", tf.__version__)

df = pd.read_csv(CSV_PATH)
df = df.fillna(0)
df.replace([np.inf, -np.inf], 0, inplace=True)

df['IN_BYTE_PER_PKT'] = df['IN_BYTES'] / (df['IN_PKTS'] + 1e-6)
df['OUT_BYTE_PER_PKT'] = df['OUT_BYTES'] / (df['OUT_PKTS'] + 1e-6)
df['TOTAL_BYTES'] = df['IN_BYTES'] + df['OUT_BYTES']
df['TOTAL_PKTS'] = df['IN_PKTS'] + df['OUT_PKTS']
df['FLOW_DUR_PER_PKT'] = df['FLOW_DURATION_MILLISECONDS'] / (df['TOTAL_PKTS'] + 1e-6)
df['IN_OUT_BYTES_RATIO'] = df['IN_BYTES'] / (df['OUT_BYTES'] + 1e-6)
df['IN_OUT_PKTS_RATIO'] = df['IN_PKTS'] / (df['OUT_PKTS'] + 1e-6)
df['TCP_SYN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x02 else 0)
df['TCP_ACK'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x10 else 0)
df['TCP_FIN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x01 else 0)

le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])
feature_cols = [c for c in df.columns if c not in ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Label', 'Attack']]

zero_day_attacks = ['scanning', 'dos', 'backdoor', 'ransomware']
df = df.reset_index(drop=True)
zero_day_df = df[df['Attack'].str.lower().isin(zero_day_attacks)].copy()
normal_df = df[~df['Attack'].str.lower().isin(zero_day_attacks)].copy()

cutoff1 = int(0.80 * len(normal_df))
dev_df = normal_df.iloc[:cutoff1].copy()
test_normal = normal_df.iloc[cutoff1:].copy()
cutoff2 = int(0.90 * len(dev_df))
train_df = dev_df.iloc[:cutoff2].copy()
val_df = dev_df.iloc[cutoff2:].copy()


def create_sequences(data, feature_cols, window_size=10, stride=5):
    sequences, labels, attacks = [], [], []
    feats = data[feature_cols].values.astype(np.float32)
    labs = data['Label'].values
    atks = data['Attack'].values
    for i in range(0, len(feats) - window_size + 1, stride):
        sequences.append(feats[i:i + window_size])
        labels.append(labs[i + window_size - 1])
        attacks.append(atks[i + window_size - 1])
    return np.array(sequences, dtype=np.float32), np.array(labels), np.array(attacks)


WINDOW, STRIDE = 10, 5
X_train, y_train, _ = create_sequences(train_df, feature_cols, WINDOW, STRIDE)
X_val, y_val, _ = create_sequences(val_df, feature_cols, WINDOW, STRIDE)
X_test_known, y_test_known, atk_known = create_sequences(test_normal, feature_cols, WINDOW, STRIDE)

fam_parts = []
for fam in zero_day_attacks:                       # each zero-day family is windowed separately
    sub = zero_day_df[zero_day_df['Attack'].str.lower() == fam]
    fam_parts.append(create_sequences(sub, feature_cols, WINDOW, STRIDE))

X_test = np.concatenate([X_test_known] + [p[0] for p in fam_parts], axis=0)
y_test = np.concatenate([y_test_known] + [p[1] for p in fam_parts], axis=0)
attack_test = np.concatenate([atk_known] + [p[2] for p in fam_parts], axis=0)

scaler = StandardScaler()                           # fitted on training windows only
X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_val = scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
X_test = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

n_s, t_, f_ = X_train.shape                         # oversampling on training windows only
X_train_2d, y_train = RandomOverSampler(random_state=42).fit_resample(X_train.reshape(n_s, -1), y_train)
X_train = X_train_2d.reshape(-1, t_, f_)
n_features = X_train.shape[-1]
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape, "| n_features =", n_features)

# ---- structural checks (hard asserts) ----
assert train_df['Attack'].str.lower().isin(zero_day_attacks).sum() == 0, "zero-day rows leaked into train"
assert val_df['Attack'].str.lower().isin(zero_day_attacks).sum() == 0, "zero-day rows leaked into validation"
assert X_train.shape[1:] == (10, n_features) and X_val.shape[1:] == (10, n_features) and X_test.shape[1:] == (10, n_features)
assert set(np.unique(y_test)) == {0, 1} and set(np.unique(y_train)) == {0, 1}

atk_lower = np.char.lower(attack_test.astype(str))
fam_masks = {a: (atk_lower == a) for a in zero_day_attacks}
known_mask = ~np.isin(atk_lower, zero_day_attacks)

# ---- comparison with the numbers reported in the paper (warnings only) ----
EXPECTED = {'scanning': 4292, 'dos': 3542, 'backdoor': 3448, 'ransomware': 27}
print("\nZero-day windows (expected values are those of the real NF-ToN-IoT file):")
for a in zero_day_attacks:
    n = int(fam_masks[a].sum())
    print(f"  {a:11s} {n:6d}   expected {EXPECTED[a]:6d}   {'OK' if n == EXPECTED[a] else 'DIFFERENT'}")
print(f"  total test windows {len(y_test)} (paper: 64216) | benign test windows {int((y_test == 0).sum())} (paper: 15281) "
      f"| features {n_features} (paper: 20)")

TensorFlow: 2.21.0
Train: (316512, 10, 20) | Val: (21162, 10, 20) | Test: (64216, 10, 20) | n_features = 20

Zero-day windows (expected values are those of the real NF-ToN-IoT file):
  scanning      4292   expected   4292   OK
  dos           3542   expected   3542   OK
  backdoor      3448   expected   3448   OK
  ransomware      27   expected     27   OK
  total test windows 64216 (paper: 64216) | benign test windows 15281 (paper: 15281) | features 20 (paper: 20)


In [3]:



class StandardESN(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate

    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1, 1, (self.units, self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units, self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        super().build(input_shape)

    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))

        def body(t, state):
            pre = tf.matmul(inputs[:, t, :], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t + 1, (1 - self.leak_rate) * state + self.leak_rate * tf.tanh(pre)
        _, final = tf.while_loop(lambda t, s: t < tf.shape(inputs)[1], body, (0, state))
        return final


class HarmonicESN(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate

    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1, 1, (self.units, self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units, self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        super().build(input_shape)

    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))

        def body(t, state):
            pre = tf.matmul(inputs[:, t, :], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t + 1, (1 - self.leak_rate) * state + self.leak_rate * tf.sin(pre)
        _, final = tf.while_loop(lambda t, s: t < tf.shape(inputs)[1], body, (0, state))
        return final


class FB_HESN_Layer(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, omega_min=0.3, omega_max=3.5, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
        self.omega_min = omega_min
        self.omega_max = omega_max

    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1, 1, (self.units, self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units, self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        omega = np.random.uniform(self.omega_min, self.omega_max, self.units).astype(np.float32)
        self.omega = self.add_weight(shape=(self.units,), initializer=tf.constant_initializer(omega), trainable=False)
        super().build(input_shape)

    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))

        def body(t, state):
            pre = tf.matmul(inputs[:, t, :], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t + 1, (1 - self.leak_rate) * state + self.leak_rate * tf.sin(self.omega * pre)
        _, final = tf.while_loop(lambda t, s: t < tf.shape(inputs)[1], body, (0, state))
        return final


# --- one generic layer that can express all variants (same random-number order as above) ---
class GenericESN(Layer):
    """act        : 'tanh' or 'sin'
       omega_mode : 'none'     -> no scaling                  (Standard / Harmonic ESN)
                    'constant' -> one shared scale omega_const (Ctrl-A)   [no extra weight]
                    'uniform'  -> per-neuron omega ~ U[omega_min, omega_max], drawn AFTER W and
                                  its mask, exactly like FB_HESN_Layer       (FB-HESN, Ctrl-B)"""

    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, act='tanh', omega_mode='none',
                 omega_const=1.0, omega_min=0.3, omega_max=1.0, **kwargs):
        super().__init__(**kwargs)
        assert act in ('tanh', 'sin') and omega_mode in ('none', 'constant', 'uniform')
        self.units, self.spectral_radius, self.leak_rate = units, spectral_radius, leak_rate
        self.act, self.omega_mode, self.omega_const = act, omega_mode, float(omega_const)
        self.omega_min, self.omega_max = omega_min, omega_max

    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1, 1, (self.units, self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units, self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        if self.omega_mode == 'uniform':
            omega = np.random.uniform(self.omega_min, self.omega_max, self.units).astype(np.float32)
            self.omega = self.add_weight(shape=(self.units,), initializer=tf.constant_initializer(omega), trainable=False)
        super().build(input_shape)

    def scale_vector(self):
        if self.omega_mode == 'uniform':
            return self.omega.numpy()
        if self.omega_mode == 'constant':
            return np.full(self.units, self.omega_const, dtype=np.float32)
        return np.ones(self.units, dtype=np.float32)

    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))

        def body(t, state):
            pre = tf.matmul(inputs[:, t, :], self.W_in) + tf.matmul(state, self.W) + self.bias
            if self.omega_mode == 'uniform':
                pre = self.omega * pre
            elif self.omega_mode == 'constant':
                pre = self.omega_const * pre
            a = tf.tanh(pre) if self.act == 'tanh' else tf.sin(pre)
            return t + 1, (1 - self.leak_rate) * state + self.leak_rate * a
        _, final = tf.while_loop(lambda t, s: t < tf.shape(inputs)[1], body, (0, state))
        return final

In [4]:
def layer_output(layer, seed=42, n=64):
    tf.keras.backend.clear_session()
    set_seed(seed)
    inp = Input(shape=(10, n_features))
    m = Model(inp, layer(inp))
    return m.predict(X_val[:n], verbose=0)


print("\nEquivalence test (max |difference| of the reservoir output, original vs generic layer):")
pairs = [
    ("Standard ESN", lambda: StandardESN(units=64), lambda: GenericESN(units=64, act='tanh', omega_mode='none')),
    ("Harmonic ESN", lambda: HarmonicESN(units=64), lambda: GenericESN(units=64, act='sin', omega_mode='none')),
    ("FB-HESN", lambda: FB_HESN_Layer(units=64, omega_min=OMEGA_MIN, omega_max=OMEGA_MAX),
     lambda: GenericESN(units=64, act='sin', omega_mode='uniform', omega_min=OMEGA_MIN, omega_max=OMEGA_MAX)),
]
equiv_ok = True
for name, make_old, make_new in pairs:
    d = float(np.max(np.abs(layer_output(make_old()) - layer_output(make_new()))))
    ok = d < 1e-5
    equiv_ok &= ok
    print(f"  {name:13s} {d:.2e}   {'PASS' if ok else 'FAIL'}")
if not equiv_ok:
    msg = "Generic layer does NOT reproduce the original layers -> results would not be comparable to the paper."
    if STRICT_EQUIVALENCE:
        raise RuntimeError(msg)
    print("WARNING:", msg)


Equivalence test (max |difference| of the reservoir output, original vs generic layer):

  Standard ESN  0.00e+00   PASS
  Harmonic ESN  0.00e+00   PASS
  FB-HESN       0.00e+00   PASS


In [5]:
MID_OMEGA = round((OMEGA_MIN + OMEGA_MAX) / 2, 4)          # 0.65, fixed a priori, never tuned


def reservoir_builder(units=64, **layer_kw):
    def build():
        inputs = Input(shape=(10, n_features))
        x = GenericESN(units=units, spectral_radius=0.9, leak_rate=0.3, omega_min=OMEGA_MIN, omega_max=OMEGA_MAX,
                       **layer_kw)(inputs)
        x = Dense(32, activation='relu')(x)
        x = Dropout(0.3)(x)
        return Model(inputs, Dense(2, activation='softmax')(x))
    return build


def build_dnn():
    inputs = Input(shape=(10, n_features))
    x = Flatten()(inputs)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.2)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))


def build_gru():
    inputs = Input(shape=(10, n_features))
    x = GRU(64)(inputs)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    return Model(inputs, Dense(2, activation='softmax')(x))


REF = 'FB-HESN-64 (omega*sin)'
MODELS = {                                                            # name -> (builder, is_reservoir)
    'Standard ESN-64 (tanh)': (reservoir_builder(act='tanh', omega_mode='none'), True),
    'Harmonic ESN-64 (shared sin)': (reservoir_builder(act='sin', omega_mode='none'), True),
    REF: (reservoir_builder(act='sin', omega_mode='uniform'), True),
}
if RUN_CONTROLS:
    MODELS[f'Ctrl-A: shared sin, scale {MID_OMEGA}'] = (
        reservoir_builder(act='sin', omega_mode='constant', omega_const=MID_OMEGA), True)
    MODELS['Ctrl-B: tanh, per-neuron omega'] = (reservoir_builder(act='tanh', omega_mode='uniform'), True)
if RUN_BASELINES:
    MODELS['GRU-64'] = (build_gru, False)
    MODELS['DNN'] = (build_dnn, False)
    MODELS['FB-HESN-128 (omega*sin)'] = (reservoir_builder(units=128, act='sin', omega_mode='uniform'), True)

In [6]:
def _act(layer):
    return np.tanh if layer.act == 'tanh' else np.sin


def reservoir_diagnostics(layer, X, n_win=1500, seed=0):
    s, f = layer.scale_vector(), _act(layer)
    Win, W, b, a = layer.W_in.numpy(), layer.W.numpy(), layer.bias.numpy(), layer.leak_rate
    rng = np.random.default_rng(seed)
    U = X[rng.choice(len(X), size=min(n_win, len(X)), replace=False)].astype(np.float32)
    x = np.zeros((U.shape[0], W.shape[0]), np.float32)
    beyond = tot = 0
    sum_abs = 0.0
    for t in range(U.shape[1]):
        arg = s * (U[:, t, :] @ Win + x @ W + b)
        beyond += int(np.sum(np.abs(arg) > np.pi / 2))
        tot += arg.size
        sum_abs += float(np.sum(np.abs(arg)))
        x = (1 - a) * x + a * f(arg)
    W_eff = W * s[None, :]                 # states are row vectors: x @ W, then scaled per neuron (column)
    return dict(
        sigma_max_W=float(np.linalg.norm(W, 2)),
        omega_mean=float(s.mean()), omega_max=float(s.max()),
        rho_W_eff=float(np.max(np.abs(np.linalg.eigvals(W_eff)))),
        sigma_max_W_eff=float(np.linalg.norm(W_eff, 2)),
        omega_max_x_sigma_max=float(s.max() * np.linalg.norm(W, 2)),
        frac_outside_first_arc=(beyond / tot if layer.act == 'sin' else np.nan),
        mean_abs_argument=sum_abs / tot,
    )


def check_scaling_equivalence(layer, X, n=64):
    """sin(s*(Win u + W x + b)) must equal plain sin with column-scaled weights (algebra check)."""
    s, f = layer.scale_vector(), _act(layer)
    Win, W, b, a = layer.W_in.numpy(), layer.W.numpy(), layer.bias.numpy(), layer.leak_rate
    U = X[:n].astype(np.float32)
    x1 = np.zeros((n, W.shape[0]), np.float32)
    x2 = np.zeros_like(x1)
    for t in range(U.shape[1]):
        x1 = (1 - a) * x1 + a * f(s * (U[:, t, :] @ Win + x1 @ W + b))
        x2 = (1 - a) * x2 + a * f(U[:, t, :] @ (Win * s) + x2 @ (W * s) + b * s)
    return float(np.max(np.abs(x1 - x2)))


def esp_check(layer, X, K=100, n_pairs=N_PAIRS, eps=1e-3, seed=0):
    """Two copies, different random start states in [-1,1], same K-step input built from scaled test windows."""
    s, f = layer.scale_vector(), _act(layer)
    Win, W, b, a = layer.W_in.numpy(), layer.W.numpy(), layer.bias.numpy(), layer.leak_rate
    rng = np.random.default_rng(seed)
    U = X[: K // 10 + 1].reshape(-1, X.shape[-1])[:K]
    worst_final, worst_step = 0.0, 0
    for _ in range(n_pairs):
        x1, x2 = rng.uniform(-1, 1, W.shape[0]), rng.uniform(-1, 1, W.shape[0])
        first_ok = None
        for t, u in enumerate(U, 1):
            x1 = (1 - a) * x1 + a * f(s * (u @ Win + x1 @ W + b))
            x2 = (1 - a) * x2 + a * f(s * (u @ Win + x2 @ W + b))
            if first_ok is None and np.max(np.abs(x1 - x2)) < eps:
                first_ok = t
        worst_final = max(worst_final, float(np.max(np.abs(x1 - x2))))
        worst_step = max(worst_step, first_ok if first_ok else K + 1)
    return worst_final, worst_step

In [7]:
def metrics_for(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return dict(f1=f1_score(y_true, y_pred, average='macro'), mcc=matthews_corrcoef(y_true, y_pred),
                fpr=fp / (fp + tn) if (fp + tn) else np.nan, fnr=fn / (fn + tp) if (fn + tp) else np.nan)


def time_inference(model):
    model.predict(X_test[:128], batch_size=256, verbose=0)                       # warm-up
    passes = []
    for _ in range(TIME_REPS):
        t0 = time.perf_counter()
        model.predict(X_test, batch_size=256, verbose=0)
        passes.append(time.perf_counter() - t0)

    @tf.function(reduce_retracing=True)
    def fwd(x):
        return model(x, training=False)
    for i in range(3):
        fwd(tf.constant(X_test[i:i + 1])).numpy()                                # trace + warm-up
    lat = []
    for i in range(LAT_CALLS):
        x = tf.constant(X_test[i:i + 1])
        t0 = time.perf_counter()
        fwd(x).numpy()
        lat.append(time.perf_counter() - t0)
    return float(np.median(passes)), float(np.median(lat) * 1e3)


def run_one(name, builder, is_reservoir, seed):
    tf.keras.backend.clear_session()
    set_seed(seed)
    model = builder()
    row = dict(model=name, seed=seed)
    diag = {}
    if is_reservoir:
        layer = [l for l in model.layers if isinstance(l, GenericESN)][0]
        diag = reservoir_diagnostics(layer, X_test)
        diag['scaling_algebra_check'] = check_scaling_equivalence(layer, X_test)
        diag['esp_diff_K10'], _ = esp_check(layer, X_test, K=10)
        diag['esp_diff_K100'], diag['esp_steps_to_eps'] = esp_check(layer, X_test, K=100)
    model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy')
    t0 = time.perf_counter()
    hist = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=MAX_EPOCHS, batch_size=256,
                     callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)],
                     verbose=0)
    train_time = time.perf_counter() - t0
    prob = model.predict(X_test, batch_size=256, verbose=0)[:, 1]
    pred = (prob >= 0.5).astype(int)
    m_all = metrics_for(y_test, pred)
    m_kn = metrics_for(y_test[known_mask], pred[known_mask])
    zd = {}
    for a in zero_day_attacks:
        det, tot = int((pred[fam_masks[a]] == 1).sum()), int(fam_masks[a].sum())
        zd[a] = (det, tot, det / tot if tot else np.nan)
    pass_s, lat_ms = time_inference(model)
    row.update({
        'macro_f1': m_all['f1'], 'mcc': m_all['mcc'], 'fpr': m_all['fpr'], 'fnr': m_all['fnr'],
        'known_macro_f1': m_kn['f1'], 'known_mcc': m_kn['mcc'], 'known_fpr': m_kn['fpr'], 'known_fnr': m_kn['fnr'], 
        'zd_macro': float(np.nanmean([zd[a][2] for a in zero_day_attacks])),
        'zd_min': float(np.nanmin([zd[a][2] for a in zero_day_attacks])),
        **{f'{a}_recall': zd[a][2] for a in zero_day_attacks},
        **{f'{a}_detected': zd[a][0] for a in zero_day_attacks},
        **{f'{a}_total': zd[a][1] for a in zero_day_attacks},
        'epochs_run': len(hist.history['loss']), 'train_time_s': train_time,
        'test_pass_s': pass_s, 'latency_b1_ms': lat_ms,
        'total_params': int(model.count_params()),
        'trainable_params': int(sum(int(np.prod(v.shape)) for v in model.trainable_weights)),
        **diag})
    return row

In [8]:
RUNS_CSV = os.path.join(OUT_DIR, 'per_run_results.csv')
runs = pd.read_csv(RUNS_CSV).to_dict('records') if os.path.exists(RUNS_CSV) else []
done = {(r['model'], int(r['seed'])) for r in runs}

for seed in SEEDS:
    for name, (builder, is_res) in MODELS.items():
        if (name, seed) in done:
            print(f"skip (already done): {name} | seed {seed}")
            continue
        t_start = time.time()
        r = run_one(name, builder, is_res, seed)
        runs.append(r)
        pd.DataFrame(runs).to_csv(RUNS_CSV, index=False)
        print(f"seed {seed} | {name:38s} F1={r['macro_f1']:.4f} ZD={r['zd_macro']:.4f} "
              f"DoS={r['dos_recall']:.3f} Back={r['backdoor_recall']:.3f} Ransom={r['ransomware_detected']}/{r['ransomware_total']} "
              f"| {time.time() - t_start:.0f}s")

runs_df = pd.DataFrame(runs)

seed 42 | Standard ESN-64 (tanh)                 F1=0.9406 ZD=0.6096 DoS=0.397 Back=0.917 Ransom=4/27 | 192s
seed 42 | Harmonic ESN-64 (shared sin)           F1=0.9330 ZD=0.5636 DoS=0.492 Back=0.704 Ransom=2/27 | 145s
seed 42 | FB-HESN-64 (omega*sin)                 F1=0.9471 ZD=0.6573 DoS=0.442 Back=0.984 Ransom=6/27 | 116s
seed 42 | Ctrl-A: shared sin, scale 0.65         F1=0.9166 ZD=0.5576 DoS=0.032 Back=0.931 Ransom=8/27 | 121s
seed 42 | GRU-64                                 F1=0.9952 ZD=0.7396 DoS=0.979 Back=0.982 Ransom=0/27 | 235s
seed 42 | DNN                                    F1=0.9793 ZD=0.7068 DoS=0.966 Back=0.790 Ransom=2/27 | 80s
seed 42 | FB-HESN-128 (omega*sin)                F1=0.9712 ZD=0.7253 DoS=0.971 Back=0.711 Ransom=6/27 | 189s
seed 43 | Standard ESN-64 (tanh)                 F1=0.9541 ZD=0.6409 DoS=0.976 Back=0.479 Ransom=3/27 | 157s
seed 43 | Harmonic ESN-64 (shared sin)           F1=0.9818 ZD=0.7821 DoS=0.969 Back=0.942 Ransom=6/27 | 100s
seed 43 | FB-HESN-64

In [9]:
PRIOR = {  # (macro_f1, zd_macro) per seed, from 64_-_comp-Omega_1 (omega in [0.3, 1.0])
    'Standard ESN-64 (tanh)': {42: (.9406, .6096), 43: (.9541, .6409), 44: (.9752, .6915), 45: (.9915, .7471), 46: (.9814, .7295)},
    'Harmonic ESN-64 (shared sin)': {42: (.9330, .5636), 43: (.9818, .7821), 44: (.9754, .6913), 45: (.9917, .7501), 46: (.9899, .7674)},
    REF: {42: (.9471, .6573), 43: (.9854, .7817), 44: (.9850, .7463), 45: (.9820, .7211), 46: (.9833, .8034)},
}
if not QUICK_TEST and (OMEGA_MIN, OMEGA_MAX) == (0.3, 1.0):
    print("\nReproduction check vs the earlier five-seed run (tolerance 0.015; small differences can come from "
          "TensorFlow / oneDNN non-determinism):")
    for name, per_seed in PRIOR.items():
        for sd, (f1_old, zd_old) in per_seed.items():
            hit = runs_df[(runs_df.model == name) & (runs_df.seed == sd)]
            if len(hit):
                d1, d2 = abs(hit.macro_f1.iloc[0] - f1_old), abs(hit.zd_macro.iloc[0] - zd_old)
                print(f"  {name:30s} seed {sd}: dF1={d1:.4f} dZD={d2:.4f}  {'OK' if max(d1, d2) < 0.015 else 'DIFFERS'}")


Reproduction check vs the earlier five-seed run (tolerance 0.015; small differences can come from TensorFlow / oneDNN non-determinism):
  Standard ESN-64 (tanh)         seed 42: dF1=0.0000 dZD=0.0000  OK
  Standard ESN-64 (tanh)         seed 43: dF1=0.0000 dZD=0.0000  OK
  Standard ESN-64 (tanh)         seed 44: dF1=0.0000 dZD=0.0000  OK
  Standard ESN-64 (tanh)         seed 45: dF1=0.0000 dZD=0.0000  OK
  Standard ESN-64 (tanh)         seed 46: dF1=0.0000 dZD=0.0000  OK
  Harmonic ESN-64 (shared sin)   seed 42: dF1=0.0000 dZD=0.0000  OK
  Harmonic ESN-64 (shared sin)   seed 43: dF1=0.0000 dZD=0.0000  OK
  Harmonic ESN-64 (shared sin)   seed 44: dF1=0.0000 dZD=0.0000  OK
  Harmonic ESN-64 (shared sin)   seed 45: dF1=0.0000 dZD=0.0000  OK
  Harmonic ESN-64 (shared sin)   seed 46: dF1=0.0000 dZD=0.0000  OK
  FB-HESN-64 (omega*sin)         seed 42: dF1=0.0000 dZD=0.0000  OK
  FB-HESN-64 (omega*sin)         seed 43: dF1=0.0000 dZD=0.0000  OK
  FB-HESN-64 (omega*sin)         seed 44: dF1=0

In [10]:
METRICS = ['macro_f1', 'mcc', 'fpr', 'zd_macro', 'zd_min', 'scanning_recall', 'dos_recall', 'backdoor_recall',
           'ransomware_recall', 'known_macro_f1', 'known_mcc', 'known_fpr']
rows_sum = []
for name, g in runs_df.groupby('model', sort=False):
    row = {'model': name, 'n_seeds': len(g)}
    for m in METRICS:
        row[f'{m}_mean'] = g[m].mean()
        row[f'{m}_sd0'] = g[m].std(ddof=0)       # population SD, as in your Table III
        row[f'{m}_sd1'] = g[m].std(ddof=1) if len(g) > 1 else np.nan
    rows_sum.append(row)
summary = pd.DataFrame(rows_sum)
summary.to_csv(os.path.join(OUT_DIR, 'summary_mean_sd.csv'), index=False)

print("\n" + "=" * 100)
print("FIVE-SEED SUMMARY (mean +- population SD, ddof=0, as in Table III)")
print("=" * 100)
for _, r in summary.iterrows():
    print(f"\n{r['model']}  (n={r['n_seeds']})")
    for m in METRICS:
        print(f"  {m:18s}: {r[m + '_mean']:.4f} +- {r[m + '_sd0']:.4f}")

try:
    from scipy import stats
except Exception:
    stats = None
PAIR_METRICS = ['zd_macro', 'macro_f1', 'mcc', 'scanning_recall', 'dos_recall', 'backdoor_recall', 'ransomware_recall']
pair_rows = []
for other in [m for m in runs_df.model.unique() if m != REF and not m.startswith(('GRU', 'DNN', 'FB-HESN-128'))]:
    a = runs_df[runs_df.model == REF].set_index('seed')
    b = runs_df[runs_df.model == other].set_index('seed')
    common = sorted(set(a.index) & set(b.index))
    if len(common) < 2:
        continue
    for m in PAIR_METRICS:
        d = (a.loc[common, m] - b.loc[common, m]).values.astype(float)
        p = float(stats.ttest_rel(a.loc[common, m], b.loc[common, m]).pvalue) if (stats and np.std(d) > 0) else np.nan
        pair_rows.append(dict(reference=REF, other=other, metric=m, n=len(common), mean_diff=d.mean(),
                              reference_higher_in=int((d > 0).sum()), paired_t_p=p))
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(os.path.join(OUT_DIR, 'paired_vs_fbhesn.csv'), index=False)
print("\n" + "=" * 100)
print(f"PAIRED DIFFERENCES: {REF} minus other (same seed = same W and W_in)")
print("=" * 100)
if len(pairs):
    print(pairs.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


FIVE-SEED SUMMARY (mean +- population SD, ddof=0, as in Table III)

Standard ESN-64 (tanh)  (n=5)
  macro_f1          : 0.9686 +- 0.0186
  mcc               : 0.9395 +- 0.0348
  fpr               : 0.0013 +- 0.0009
  zd_macro          : 0.6837 +- 0.0520
  zd_min            : 0.0815 +- 0.0432
  scanning_recall   : 0.9915 +- 0.0078
  dos_recall        : 0.8361 +- 0.2234
  backdoor_recall   : 0.8258 +- 0.1913
  ransomware_recall : 0.0815 +- 0.0432
  known_macro_f1    : 0.9934 +- 0.0016
  known_mcc         : 0.9869 +- 0.0032
  known_fpr         : 0.0013 +- 0.0009

Harmonic ESN-64 (shared sin)  (n=5)
  macro_f1          : 0.9744 +- 0.0215
  mcc               : 0.9507 +- 0.0399
  fpr               : 0.0010 +- 0.0002
  zd_macro          : 0.7109 +- 0.0799
  zd_min            : 0.0963 +- 0.0687
  scanning_recall   : 0.9937 +- 0.0048
  dos_recall        : 0.8743 +- 0.1913
  backdoor_recall   : 0.8792 +- 0.1163
  ransomware_recall : 0.0963 +- 0.0687
  known_macro_f1    : 0.9923 +- 0.0027
  know

In [11]:
timing = runs_df.groupby('model', sort=False).agg(
    total_params=('total_params', 'first'), trainable_params=('trainable_params', 'first'),
    train_time_s=('train_time_s', 'median'), epochs_run=('epochs_run', 'median'),
    test_pass_s=('test_pass_s', 'median'), latency_b1_ms=('latency_b1_ms', 'median')).reset_index()
timing.to_csv(os.path.join(OUT_DIR, 'timing_table.csv'), index=False)
print("\n" + "=" * 100)
print("TIMING (median over seeds; one session, one machine; test_pass_s = median of "
      f"{TIME_REPS} full-test passes, batch 256; latency = one window, batch 1, graph mode)")
print("=" * 100)
print(timing.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

diag_cols = ['sigma_max_W', 'omega_mean', 'omega_max', 'rho_W_eff', 'sigma_max_W_eff', 'omega_max_x_sigma_max',
             'frac_outside_first_arc', 'mean_abs_argument', 'scaling_algebra_check',
             'esp_diff_K10', 'esp_diff_K100', 'esp_steps_to_eps']
have = [c for c in diag_cols if c in runs_df.columns]
diag_df = runs_df[['model', 'seed'] + have].dropna(subset=[have[0]]) if have else pd.DataFrame()
diag_df.to_csv(os.path.join(OUT_DIR, 'reservoir_diagnostics.csv'), index=False)
if len(diag_df):
    print("\n" + "=" * 100)
    print("RESERVOIR DIAGNOSTICS (mean over seeds)  -- frac_outside_first_arc = share of |omega*z| > pi/2")
    print("=" * 100)
    print(diag_df.groupby('model', sort=False)[have].mean().to_string(float_format=lambda v: f"{v:.4g}"))
    print(f"\nlargest scaling-algebra mismatch: {diag_df['scaling_algebra_check'].max():.2e} (should be ~1e-6 or less)")
    print(f"largest echo-state difference after K=100: {diag_df['esp_diff_K100'].max():.2e} (tolerance 1e-3); "
          f"max steps to tolerance: {int(diag_df['esp_steps_to_eps'].max())}")

with open(os.path.join(OUT_DIR, 'config.json'), 'w') as fh:
    json.dump(dict(seeds=SEEDS, omega=[OMEGA_MIN, OMEGA_MAX], ctrl_a_scale=MID_OMEGA, max_epochs=MAX_EPOCHS,
                   tensorflow=tf.__version__, quick_test=QUICK_TEST, n_features=int(n_features),
                   test_windows=int(len(y_test))), fh, indent=2)
print(f"\nDone. Files written to: {os.path.abspath(OUT_DIR)}")


TIMING (median over seeds; one session, one machine; test_pass_s = median of 5 full-test passes, batch 256; latency = one window, batch 1, graph mode)
                         model  total_params  trainable_params  train_time_s  epochs_run  test_pass_s  latency_b1_ms
        Standard ESN-64 (tanh)          7586              2146       154.168      32.000        0.906          0.860
  Harmonic ESN-64 (shared sin)          7586              2146       136.762      25.000        1.057          0.921
        FB-HESN-64 (omega*sin)          7650              2146       185.992      33.000        1.020          0.943
Ctrl-A: shared sin, scale 0.65          7586              2146       156.447      33.000        0.974          0.860
Ctrl-B: tanh, per-neuron omega          7650              2146       116.975      27.000        0.912          0.890
                        GRU-64         18658             18658       183.685       9.000        1.661          1.135
                           DN

In [12]:
import pandas as pd, numpy as np

runs = pd.read_csv(RUNS_CSV)
N_TOT = len(y_test) if 'y_test' in globals() else 64216               # test windows (paper: 64216)
N_BEN = int((y_test == 0).sum()) if 'y_test' in globals() else 15281  # benign test windows (paper: 15281)
N_ATT = N_TOT - N_BEN
print("test windows:", N_TOT, "| benign:", N_BEN, "| attack:", N_ATT)

# accuracy = 1 - (FP + FN)/N, with FP = FPR*N_benign and FN = FNR*N_attack  (exact for each run)
runs['accuracy'] = 1 - (runs['fpr'] * N_BEN + runs['fnr'] * N_ATT) / N_TOT

ROWS = [('Standard ESN-64 (tanh)',          r'Standard ESN-64 ($\tanh$)'),
        ('Harmonic ESN-64 (shared sin)',    r'Harmonic ESN-64 (shared $\sin$)'),
        ('FB-HESN-64 (omega*sin)',          r'FB-HESN-64 ($\omega\odot\sin$)'),
        ('Ctrl-A: shared sin, scale 0.65',  r'Ctrl-A ($\sin(0.65\,z)$)'),
        ('Ctrl-B: tanh, per-neuron omega',  r'Ctrl-B ($\tanh(\omega_i z)$)')]
COLS = ['accuracy', 'macro_f1', 'zd_macro', 'scanning_recall', 'dos_recall', 'backdoor_recall', 'ransomware_recall']

stat = {m: {c: (g[c].mean(), g[c].std(ddof=0)) for c in COLS} for m, g in runs.groupby('model')}
best = {c: round(max(stat[m][c][0] for m, _ in ROWS), 4) for c in COLS}      # higher is better in every column

print("\n% ---- Table II body (paste between \\hline rules) ----")
for model, label in ROWS:
    cells = []
    for c in COLS:
        mu, sd = stat[model][c]
        s = f"{mu:.4f}\\pm{sd:.4f}"
        cells.append(f"$\\mathbf{{{s}}}$" if round(mu, 4) == best[c] else f"${s}$")
    print(label + " & " + " & ".join(cells) + r" \\")

test windows: 64216 | benign: 15281 | attack: 48935

% ---- Table II body (paste between \hline rules) ----
Standard ESN-64 ($\tanh$) & $0.9762\pm0.0145$ & $0.9686\pm0.0186$ & $0.6837\pm0.0520$ & $0.9915\pm0.0078$ & $0.8361\pm0.2234$ & $0.8258\pm0.1913$ & $0.0815\pm0.0432$ \\
Harmonic ESN-64 (shared $\sin$) & $0.9806\pm0.0169$ & $0.9744\pm0.0215$ & $0.7109\pm0.0799$ & $0.9937\pm0.0048$ & $0.8743\pm0.1913$ & $0.8792\pm0.1163$ & $0.0963\pm0.0687$ \\
FB-HESN-64 ($\omega\odot\sin$) & $\mathbf{0.9824\pm0.0115}$ & $\mathbf{0.9766\pm0.0148}$ & $\mathbf{0.7420\pm0.0509}$ & $0.9934\pm0.0064$ & $0.8508\pm0.2059$ & $\mathbf{0.9607\pm0.0431}$ & $\mathbf{0.1630\pm0.0798}$ \\
Ctrl-A ($\sin(0.65\,z)$) & $0.9714\pm0.0216$ & $0.9628\pm0.0271$ & $0.6886\pm0.0830$ & $0.9911\pm0.0101$ & $0.7604\pm0.3673$ & $0.8400\pm0.2061$ & $\mathbf{0.1630\pm0.0864}$ \\
Ctrl-B ($\tanh(\omega_i z)$) & $0.9817\pm0.0068$ & $0.9755\pm0.0089$ & $0.7309\pm0.0416$ & $\mathbf{0.9951\pm0.0035}$ & $\mathbf{0.8981\pm0.1420}$ & $0.

In [13]:
import pandas as pd
d = pd.read_csv('fbhesn_extra_results/reservoir_diagnostics.csv')
print(d.esp_diff_K10.max().round(2), d.esp_diff_K100.max(), d.esp_steps_to_eps.max())

0.95 0.0001667926835175 80.0
